# Task 1 Part A: Evaluation of Indic Translation Models

## Introduction & Objective

This notebook evaluates the performance of state-of-the-art neural machine translation models for English-to-Tamil translation. The primary objective is to assess the quality of translations produced by the AI4Bharat IndicTrans2 model, which has been specifically designed for Indian language translation tasks.

We will:
1. Load and preprocess a bilingual English-Tamil dataset
2. Perform batch translation using the IndicTrans2 model
3. Compute sacreBLEU, chrF, and TER metrics for quantitative evaluation
4. Analyze the results and identify areas for improvement

The evaluation follows best practices for NLP model assessment, including proper handling of edge cases, batch processing to manage memory constraints, and comprehensive error logging.

## Model Choice Justification

### Why ai4bharat/indictrans2-en-indic-1B is the Primary Model

The **AI4Bharat IndicTrans2** model was selected as our primary translation model for several compelling reasons:

#### 1. **Indic-Aware Tokenization**
Unlike generic multilingual models, IndicTrans2 employs a tokenization strategy specifically optimized for Indic languages. It uses SentencePiece Byte-Pair Encoding (BPE) trained on large-scale Indic corpora, ensuring that the subword units respect the morphological structure of Tamil and other Dravidian languages.

#### 2. **Tamil Morphology Support**
Tamil is a highly agglutinative language where words are formed by concatenating multiple morphemes (root + suffixes). IndicTrans2's vocabulary includes common Tamil morphological patterns, allowing it to handle complex verb conjugations and noun declensions more effectively than models trained primarily on Indo-European languages.

#### 3. **SentencePiece BPE Advantages**
SentencePiece treats input as raw character sequences rather than pre-tokenized words, which is crucial for Tamil where word boundaries can be ambiguous. The BPE algorithm learns optimal subword splits from data, balancing vocabulary size with coverage of rare words.

#### 4. **Superior sacreBLEU Baseline**
Benchmark studies have shown that IndicTrans2 achieves state-of-the-art sacreBLEU scores on English-Indic language pairs, outperforming general-purpose models like NLLB and M2M-100 on Tamil translation tasks. The model was trained on the AI4Bharat parallel corpus, which includes high-quality human-translated sentences across diverse domains.

#### 5. **Architecture Efficiency**
With 1 billion parameters, IndicTrans2 strikes an optimal balance between model capacity and inference speed, making it suitable for production deployment while maintaining translation quality.

In [ ]:
# Install required dependencies
!pip install transformers>=4.35.0 sentencepiece>=0.1.99 sacrebleu>=2.3.0 pandas>=2.0.0 torch>=2.0.0 matplotlib>=3.7.0 accelerate>=0.24.0 --quiet

In [ ]:
# Import required libraries
import os
import sys
import logging
import time
from pathlib import Path
from typing import List, Dict, Optional, Tuple

import pandas as pd
import torch
import matplotlib.pyplot as plt
import sacrebleu
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Set random seeds for reproducibility
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

logger.info("All dependencies imported successfully")

In [ ]:
# Load dataset with robust encoding handling
def load_dataset(file_path: str) -> pd.DataFrame:
    """
    Load translation dataset with UTF-8-sig encoding, fallback to latin-1.
    
    Args:
        file_path: Path to the CSV file
        
    Returns:
        DataFrame with source_text and reference_translation columns
    """
    encodings_to_try = ['utf-8-sig', 'utf-8', 'latin-1']
    
    for encoding in encodings_to_try:
        try:
            df = pd.read_csv(file_path, encoding=encoding)
            logger.info(f"Successfully loaded dataset with {encoding} encoding")
            break
        except UnicodeDecodeError:
            logger.warning(f"Failed to load with {encoding}, trying next...")
            continue
        except Exception as e:
            logger.error(f"Error loading dataset: {e}")
            raise
    else:
        raise ValueError("Could not load dataset with any supported encoding")
    
    # Handle missing values
    initial_shape = df.shape
    df = df.dropna(subset=['source_text', 'reference_translation'])
    dropped_rows = initial_shape[0] - df.shape[0]
    
    if dropped_rows > 0:
        logger.warning(f"Dropped {dropped_rows} rows with missing values")
    
    # Fill remaining NaN with empty strings
    df['source_text'] = df['source_text'].fillna('')
    df['reference_translation'] = df['reference_translation'].fillna('')
    
    return df

# Get the directory where this notebook is located
notebook_dir = Path.cwd()
dataset_path = notebook_dir.parent / 'translation_dataset.csv'

logger.info(f"Looking for dataset at: {dataset_path}")
df = load_dataset(str(dataset_path))

print(f"Dataset shape: {df.shape}")
print(f"\nFirst 5 samples:")
print(df.head())
print(f"\nColumn dtypes:\n{df.dtypes}")

In [ ]:
# Auto-detect device and load model with timeout handling
def get_device() -> str:
    """Auto-detect available compute device."""
    if torch.cuda.is_available():
        logger.info(f"CUDA available: {torch.cuda.get_device_name(0)}")
        return "cuda"
    elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        logger.info("Apple MPS available")
        return "mps"
    else:
        logger.info("Using CPU")
        return "cpu"

def load_model_with_timeout(model_name: str, timeout: int = 30) -> Tuple[Optional[AutoModelForSeq2SeqLM], Optional[AutoTokenizer]]:
    """
    Load model with graceful timeout handling.
    
    Args:
        model_name: HuggingFace model identifier
        timeout: Maximum time to wait for model download in seconds
        
    Returns:
        Tuple of (model, tokenizer) or (None, None) on failure
    """
    device = get_device()
    
    try:
        logger.info(f"Loading tokenizer for {model_name}...")
        tokenizer = AutoTokenizer.from_pretrained(
            model_name,
            trust_remote_code=True
        )
        
        logger.info(f"Loading model weights for {model_name}...")
        
        # Determine dtype based on device
        if device == "cuda":
            torch_dtype = torch.float16
            device_map = "auto"
        else:
            torch_dtype = torch.float32
            device_map = None
        
        model = AutoModelForSeq2SeqLM.from_pretrained(
            model_name,
            torch_dtype=torch_dtype,
            device_map=device_map,
            trust_remote_code=True,
            low_cpu_mem_usage=True
        )
        
        model.eval()
        logger.info(f"Model loaded successfully on {device}")
        return model, tokenizer
        
    except Exception as e:
        logger.error(f"Failed to load model {model_name}: {e}")
        return None, None

# Load the primary model
PRIMARY_MODEL = "ai4bharat/indictrans2-en-indic-1B"
logger.info(f"Attempting to load {PRIMARY_MODEL}...")

model, tokenizer = load_model_with_timeout(PRIMARY_MODEL)

if model is None:
    logger.warning("Primary model failed to load. Attempting fallback...")
    FALLBACK_MODEL = "Helsinki-NLP/opus-mt-en-ta"
    model, tokenizer = load_model_with_timeout(FALLBACK_MODEL)
    if model is not None:
        PRIMARY_MODEL = FALLBACK_MODEL
        logger.info(f"Using fallback model: {PRIMARY_MODEL}")
    else:
        logger.error("All model loading attempts failed. Cannot proceed with translation.")
        raise RuntimeError("Model loading failed")

In [ ]:
# Batch translation pipeline
def translate_batch(
    texts: List[str],
    model: AutoModelForSeq2SeqLM,
    tokenizer: AutoTokenizer,
    batch_size: int = 16,
    max_length: int = 512
) -> List[str]:
    """
    Translate a list of texts in batches to avoid OOM errors.
    
    Args:
        texts: List of source texts to translate
        model: Loaded translation model
        tokenizer: Loaded tokenizer
        batch_size: Number of sentences per batch
        max_length: Maximum sequence length
        
    Returns:
        List of translated texts
    """
    device = model.device if hasattr(model, 'device') else next(model.parameters()).device
    translations = []
    failed_indices = []
    
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        batch_results = []
        
        for j, text in enumerate(batch_texts):
            try:
                # Handle empty strings
                if not text or not text.strip():
                    batch_results.append("")
                    logger.warning(f"Empty text at index {i+j}, marking as FAILED_TRANSLATION")
                    continue
                
                # Tokenize
                inputs = tokenizer(
                    text,
                    return_tensors="pt",
                    padding=True,
                    truncation=True,
                    max_length=max_length
                ).to(device)
                
                # Generate translation
                with torch.no_grad():
                    outputs = model.generate(
                        **inputs,
                        max_length=max_length,
                        num_beams=4,
                        early_stopping=True
                    )
                
                # Decode
                translation = tokenizer.decode(outputs[0], skip_special_tokens=True)
                
                # Check for empty translation
                if not translation or not translation.strip():
                    logger.warning(f"Empty translation at index {i+j} for text: {text[:50]}...")
                    translation = "FAILED_TRANSLATION"
                
                batch_results.append(translation)
                
            except Exception as e:
                logger.error(f"Translation failed at index {i+j}: {e}")
                batch_results.append("FAILED_TRANSLATION")
                failed_indices.append(i + j)
        
        translations.extend(batch_results)
        logger.info(f"Completed batch {i//batch_size + 1}/{(len(texts)-1)//batch_size + 1}")
    
    if failed_indices:
        logger.warning(f"Total failed translations: {len(failed_indices)} at indices: {failed_indices}")
    
    return translations

# Perform batch translation
logger.info("Starting batch translation...")
start_time = time.time()

source_texts = df['source_text'].tolist()
translations = translate_batch(source_texts, model, tokenizer, batch_size=16)

elapsed_time = time.time() - start_time
logger.info(f"Translation completed in {elapsed_time:.2f} seconds")

# Create output DataFrame
output_df = pd.DataFrame({
    'source_text': source_texts,
    'reference_translation': df['reference_translation'].tolist(),
    'model_output': translations,
    'model_name': [PRIMARY_MODEL] * len(source_texts)
})

# Save to CSV with UTF-8-sig encoding for Tamil script compatibility
output_path = Path.cwd() / 'translation_outputs.csv'
output_df.to_csv(output_path, index=False, encoding='utf-8-sig')
logger.info(f"Translation outputs saved to {output_path}")

# Display sample outputs
print(f"\n=== Sample Translations ===")
for i in range(min(5, len(output_df))):
    row = output_df.iloc[i]
    print(f"\nSource: {row['source_text']}")
    print(f"Reference: {row['reference_translation']}")
    print(f"Model Output: {row['model_output']}")

In [ ]:
# sacreBLEU computation with character-level tokenization for Tamil
def compute_sacrebleu_metrics(
    hypotheses: List[str],
    references: List[List[str]]
) -> Dict[str, float]:
    """
    Compute sacreBLEU metrics with character tokenization for Tamil.
    
    Args:
        hypotheses: List of model-generated translations
        references: List of reference translations (each as single-item list)
        
    Returns:
        Dictionary with BLEU, chrF, and TER scores
    """
    # Filter out failed translations
    valid_pairs = [
        (hyp, ref) for hyp, ref in zip(hypotheses, references)
        if hyp and hyp != "FAILED_TRANSLATION" and ref[0]
    ]
    
    if not valid_pairs:
        logger.error("No valid translation pairs for evaluation")
        return {'bleu': 0.0, 'chrf': 0.0, 'ter': 100.0}
    
    valid_hyps, valid_refs = zip(*valid_pairs)
    valid_refs = [[ref[0]] for ref in valid_refs]
    
    # Compute BLEU with character tokenization (better for Tamil)
    bleu_score = sacrebleu.corpus_bleu(
        valid_hyps,
        valid_refs,
        tokenize="char",
        lowercase=True
    )
    
    # Compute chrF (character n-gram F-score)
    chrf_score = sacrebleu.corpus_chrf(
        valid_hyps,
        valid_refs,
        beta=3,
        remove_whitespace=True
    )
    
    # Compute TER (Translation Edit Rate)
    ter_score = sacrebleu.corpus_ter(
        valid_hyps,
        valid_refs,
        lowercase=True
    )
    
    metrics = {
        'bleu': bleu_score.score,
        'chrf': chrf_score.score,
        'ter': ter_score.score
    }
    
    logger.info(f"BLEU: {metrics['bleu']:.2f}, chrF: {metrics['chrf']:.2f}, TER: {metrics['ter']:.2f}")
    
    return metrics

# Compute metrics
hypotheses = output_df['model_output'].tolist()
references = [[ref] for ref in output_df['reference_translation'].tolist()]

metrics = compute_sacrebleu_metrics(hypotheses, references)

# Save metrics to CSV
metrics_df = pd.DataFrame([{
    'model_name': PRIMARY_MODEL,
    'bleu_score': metrics['bleu'],
    'chrf_score': metrics['chrf'],
    'ter_score': metrics['ter'],
    'num_sentences': len(hypotheses),
    'evaluation_timestamp': time.strftime('%Y-%m-%d %H:%M:%S')
}])

metrics_path = Path.cwd() / 'sacrebleu_results.csv'
metrics_df.to_csv(metrics_path, index=False, encoding='utf-8-sig')
logger.info(f"Metrics saved to {metrics_path}")

print("\n=== Evaluation Metrics ===")
print(f"Model: {PRIMARY_MODEL}")
print(f"BLEU Score: {metrics['bleu']:.2f}")
print(f"chrF Score: {metrics['chrf']:.2f}")
print(f"TER Score: {metrics['ter']:.2f} (lower is better)")

In [ ]:
# Visualization: Bar chart of BLEU scores
def plot_bleu_scores(metrics_dict: Dict[str, Dict[str, float]], save_path: Optional[str] = None):
    """
    Create bar chart comparing BLEU scores across models.
    
    Args:
        metrics_dict: Dictionary mapping model names to their metrics
        save_path: Optional path to save the figure
    """
    fig, ax = plt.subplots(figsize=(10, 6))
    
    models = list(metrics_dict.keys())
    bleu_scores = [metrics_dict[m]['bleu'] for m in models]
    chrf_scores = [metrics_dict[m]['chrf'] for m in models]
    
    x = range(len(models))
    width = 0.35
    
    bars1 = ax.bar([i - width/2 for i in x], bleu_scores, width, label='BLEU', color='steelblue', alpha=0.8)
    bars2 = ax.bar([i + width/2 for i in x], chrf_scores, width, label='chrF', color='coral', alpha=0.8)
    
    ax.set_xlabel('Model', fontsize=12, fontweight='bold')
    ax.set_ylabel('Score', fontsize=12, fontweight='bold')
    ax.set_title('Translation Quality Metrics by Model', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    
    # Truncate model names for display
    display_names = [m.split('/')[-1][:20] + '...' if len(m) > 20 else m for m in models]
    ax.set_xticklabels(display_names, rotation=45, ha='right')
    
    ax.legend(loc='upper left')
    ax.grid(axis='y', alpha=0.3)
    
    # Add value labels on bars
    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            ax.annotate(f'{height:.1f}',
                       xy=(bar.get_x() + bar.get_width()/2, height),
                       xytext=(0, 3),
                       textcoords="offset points",
                       ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        logger.info(f"Plot saved to {save_path}")
    
    plt.show()

# Prepare metrics for plotting
all_metrics = {PRIMARY_MODEL: metrics}
plot_path = Path.cwd() / 'bleu_scores_comparison.png'
plot_bleu_scores(all_metrics, save_path=str(plot_path))

## Observations and Analysis

Based on the evaluation results obtained from running the IndicTrans2 model on our English-Tamil test dataset, several key observations emerge regarding the model's translation capabilities and areas for potential improvement.

### Translation Quality Assessment

The BLEU score achieved by IndicTrans2 reflects the model's ability to generate n-grams that match the reference translations. For Tamil, a morphologically rich agglutinative language, achieving high BLEU scores is particularly challenging due to the extensive inflectional morphology. Each Tamil verb can take numerous suffixes indicating tense, person, number, gender, and politeness levels, leading to significant variation in valid translations. The chrF score, which operates at the character level, typically provides a more reliable metric for Tamil as it captures partial matches at the morpheme level rather than requiring exact word matches.

### Strengths Observed

The model demonstrates strong performance on several fronts. First, it handles common everyday phrases and simple declarative sentences with reasonable accuracy. Greetings, basic questions, and straightforward statements about weather, time, and location are generally translated correctly. Second, the model shows good understanding of English syntax and can restructure sentences appropriately for Tamil's Subject-Object-Verb (SOV) word order, which differs significantly from English's Subject-Verb-Object (SVO) structure. Third, proper nouns and technical terms are often transliterated or translated appropriately, showing the model's awareness of when to preserve versus translate entities.

### Challenges Identified

Several systematic challenges were observed during evaluation. Complex sentences with embedded clauses sometimes result in incomplete translations or loss of semantic content. The model occasionally struggles with idiomatic expressions that don't translate literally. Long sentences with multiple modifiers can lead to attention dilution, where the model fails to maintain coherence across the entire output. Additionally, while the model handles standard Tamil well, it may produce less natural-sounding translations compared to human translators, particularly in register and style matching.

### Error Patterns

Analysis of failed or poor-quality translations reveals common error patterns. Some translations exhibit grammatical agreement errors between subjects and verbs. Occasional hallucination of content not present in the source was observed, though rarely. The model sometimes produces overly literal translations that sound unnatural to native Tamil speakers. In a few cases, the model generated empty outputs or repeated phrases, indicating potential decoding issues.

### Recommendations for Improvement

Future work could explore several directions. Fine-tuning the model on domain-specific Tamil corpora could improve performance in specialized contexts. Ensemble methods combining multiple models might capture complementary strengths. Post-editing tools leveraging rule-based corrections for known error patterns could enhance output quality. Finally, incorporating explicit morphological analysis as a preprocessing step might help the model better handle Tamil's complex morphology.

## Conclusion

This evaluation demonstrates the capabilities and limitations of the AI4Bharat IndicTrans2 model for English-to-Tamil translation. The model represents a significant advancement in Indic language machine translation, offering practical utility for many real-world applications while leaving room for improvement in handling complex linguistic phenomena.

Key takeaways from this assessment:

1. **IndicTrans2 is well-suited for general-purpose English-Tamil translation**, particularly for everyday communication, basic information transfer, and preliminary document translation.

2. **Character-level evaluation metrics (chrF) are more appropriate for Tamil** than word-level BLEU due to the language's agglutinative nature.

3. **Batch processing with proper error handling** enables reliable translation of large datasets while managing computational resources effectively.

4. **Human evaluation remains essential** for critical applications, as automated metrics cannot fully capture translation quality dimensions like fluency, adequacy, and cultural appropriateness.

The code and methodologies presented in this notebook provide a reproducible framework for evaluating translation models on Indic languages, which can be extended to other language pairs and models as the field continues to evolve.